# Consensus V2 Training (YCrCb) — все камеры train

Цель: обучать модель на **всех** train-сэмплах (front/rear/left/right), не только side.

## Входы модели (7 каналов, YCrCb)
| Канал | Источник | Цветовая обработка |
|------|----------|-------------------|
| `warp_ycc` (3) | `side_warp_mix_v1/train/<id>/warp_mix.jpg` или `consensus_raw.npy` | Telea expand → prefill → med5 outlier_soft → no-trust 50/50 mix → ego overlay |
| `lidar_trust` (1) | `precomputed_lidar_trust/train/<id>/lidar_trust.npy` | без изменения (camera space) |
| `mean_t0t1_ycc` (3) | `input/t0,t1` | RGB → YCrCb |

## Mirror policy
- Хранение на диске: **camera-space**
- Flip H только для `left_fwd` / `right_bwd` при формировании тензора в модель
- На инференсе: разворот выхода обратно в camera-space

## Проверка инпутов
В `__getitem__` возвращается `audit` — список применённых шагов и флаги загрузки.


In [1]:
from __future__ import annotations

import json
import random
import sys
from dataclasses import dataclass, field
from pathlib import Path

import cv2
import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset

REPO = Path(r"C:/Users/adel/Documents/GitHub/YA-")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from consensus_kit import (
    ConsensusConfig,
    ConsensusUNet,
    _chw_npy_to_hwc_u8,
    _edge_contrast_mask,
    _load_ego_mask,
    _local_median_rgb01,
    _resize_hw,
    _resize_maps,
    selective_median_outlier_fix,
    discover_samples,
)
from lib.lidar_density_mask import lidar_rife_blend_maps
from ya_paths import BAKED_ROOT, CV_ROOT, EGO_MASKS_APPROVED, TRAIN_SPLIT, WARPS_ROOT


In [2]:
MIRROR_CAMERAS = ("left_fwd", "right_bwd")
WARP_MIX_ROOT = CV_ROOT / "side_warp_mix_v1" / "train"

@dataclass
class V2TrainCfg:
    image_h: int = 544
    image_w: int = 1024
    in_channels: int = 7
    base_channels: int = 32
    batch_size: int = 12  # при OOM: 2; на 24GB можно 8–12
    val_batch_size: int = 12
    num_workers: int = 0
    lr: float = 2e-4
    epochs: int = 200
    val_fraction: float = 0.16
    seed: int = 42
    trust_zone_min: float = 0.12
    trust_cache_root: Path = CV_ROOT / "precomputed_lidar_trust" / "train"
    dataset_root: Path = TRAIN_SPLIT
    baked_root: Path = BAKED_ROOT / "train"
    warps_root: Path = WARPS_ROOT / "train"
    warp_mix_root: Path = WARP_MIX_ROOT
    ego_masks_root: Path = EGO_MASKS_APPROVED
    rife_root: Path = CV_ROOT / "rife_predictions_v5" / "train"
    max_samples: int = 0  # 0 = все ready сэмплы

CFG = V2TrainCfg()
DS_CFG = ConsensusConfig(
    dataset_root=str(CFG.dataset_root),
    consensus_root=str(CFG.warps_root),
    baked_root=str(CFG.baked_root),
    rife_root=str(CFG.rife_root),
    ego_masks_root=str(CFG.ego_masks_root),
    allowed_cameras=(),  # все камеры
    image_h=CFG.image_h,
    image_w=CFG.image_w,
)
print(CFG)


V2TrainCfg(image_h=544, image_w=1024, in_channels=7, base_channels=32, batch_size=12, val_batch_size=12, num_workers=0, lr=0.0002, epochs=200, val_fraction=0.16, seed=42, trust_zone_min=0.12, trust_cache_root=WindowsPath('C:/Users/adel/Downloads/cv_dataset/precomputed_lidar_trust/train'), dataset_root=WindowsPath('C:/Users/adel/Downloads/cv_dataset/final_dataset_v5_participants/train'), baked_root=WindowsPath('C:/Users/adel/Downloads/cv_dataset/rife_refinement_baked/train'), warps_root=WindowsPath('C:/Users/adel/Downloads/cv_dataset/multiview_warps/train'), warp_mix_root=WindowsPath('C:/Users/adel/Downloads/cv_dataset/side_warp_mix_v1/train'), ego_masks_root=WindowsPath('C:/Users/adel/Documents/GitHub/YA-/methods_gallery/_ego_manual_masks/masks_approved'), rife_root=WindowsPath('C:/Users/adel/Downloads/cv_dataset/rife_predictions_v5/train'), max_samples=0)


In [3]:
from importlib.machinery import SourceFileLoader

_pwm_path = REPO / "scripts/training/precompute_warp_notrust_mix.py"
pwm = SourceFileLoader("pwm", str(_pwm_path)).load_module()
MIX = pwm.MixCfg()


def collate_v2_batch(batch):
    out = {}
    for k in batch[0]:
        if k in ("audit", "sample_id", "camera", "mirrored"):
            out[k] = [b[k] for b in batch]
        else:
            out[k] = torch.utils.data.default_collate([b[k] for b in batch])
    return out


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def _flip_h(*arrays):
    return tuple(a[:, ::-1].copy() for a in arrays)


def _to_ycrcb01(rgb_u8: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2YCrCb).astype(np.float32) / 255.0


def _ycrcb01_to_rgb01(ycc01: np.ndarray) -> np.ndarray:
    u8 = np.clip(ycc01 * 255.0, 0, 255).astype(np.uint8)
    return cv2.cvtColor(u8, cv2.COLOR_YCrCb2RGB).astype(np.float32) / 255.0


def _load_rgb(path: Path, hw: tuple[int, int]) -> np.ndarray:
    arr = np.array(Image.open(path).convert("RGB"), dtype=np.uint8)
    if arr.shape[:2] != hw:
        arr = _resize_hw(arr, hw, cv2.INTER_LINEAR)
    return arr


def _load_warp_camera_rgb(
    cfg: V2TrainCfg,
    ds_cfg: ConsensusConfig,
    sid: str,
    cam: str,
    src: Path,
    hw: tuple[int, int],
) -> tuple[np.ndarray, dict]:
    """Загрузка warp в camera-space (как precompute_warp_notrust_mix)."""
    mix_path = cfg.warp_mix_root / sid / "warp_mix.jpg"
    audit = {
        "warp_source": None,
        "warp_steps": [],
        "paths": {
            "warp_mix": str(mix_path),
            "consensus_raw": str(cfg.warps_root / sid / "consensus_raw.npy"),
            "coverage": str(cfg.warps_root / sid / "coverage.npy"),
        },
    }

    if mix_path.is_file():
        warp_u8 = _load_rgb(mix_path, hw)
        audit["warp_source"] = "side_warp_mix_v1/warp_mix.jpg"
        audit["warp_steps"] = ["load_precomputed_camera_space"]
        return warp_u8, audit

    warp_file = cfg.warps_root / sid / "consensus_raw.npy"
    cov_file = cfg.warps_root / sid / "coverage.npy"
    if not (warp_file.is_file() and cov_file.is_file()):
        raise FileNotFoundError(f"missing warp for {sid}")

    warp_u8 = _chw_npy_to_hwc_u8(warp_file)
    cov = np.load(cov_file).astype(np.float32)
    warp_u8, cov, _ = _resize_maps(warp_u8, cov, cov, hw)

    t0 = _load_rgb(src / "input" / "t0" / f"{cam}.jpg", hw)
    t1 = _load_rgb(src / "input" / "t1" / f"{cam}.jpg", hw)
    mean_u8 = ((t0.astype(np.float32) + t1.astype(np.float32)) * 0.5).astype(np.uint8)
    art = _load_ego_mask(ds_cfg, sid, cam, hw).astype(np.float32)

    trust_path = cfg.trust_cache_root / sid / "lidar_trust.npy"
    if trust_path.is_file():
        trust = np.load(trust_path).astype(np.float32)
        if trust.shape != hw:
            trust = cv2.resize(trust, (hw[1], hw[0]), interpolation=cv2.INTER_NEAREST)
    else:
        maps = lidar_rife_blend_maps(
            src, cam, hw, spread_radius_fine=3.0, spread_blur_fine=1.0, zone_min=cfg.trust_zone_min
        )
        trust = maps["lidar_trust"].astype(np.float32)
        trust_path.parent.mkdir(parents=True, exist_ok=True)
        np.save(trust_path, trust)

    warp = warp_u8.astype(np.float32) / 255.0
    mean = mean_u8.astype(np.float32) / 255.0
    mirrored_proc = cam in MIRROR_CAMERAS
    if mirrored_proc:
        warp, cov, mean, art, trust = _flip_h(warp, cov, mean, art, trust)

    seed = abs(hash(sid)) % (2**31 - 1)
    core = pwm._final_variant_n030(warp, cov, MIX, seed)
    audit["warp_steps"] = [
        "load_consensus_raw.npy",
        f"telea_expand r={MIX.warp_expand_px}",
        f"prefill noise={MIX.prefill_noise_std}",
        "telea_med5_outlier_soft",
    ]

    no_trust = (trust <= MIX.no_trust_thr).astype(np.float32)
    if MIX.no_trust_feather_px > 0:
        no_trust = cv2.GaussianBlur(
            no_trust, (0, 0), sigmaX=MIX.no_trust_feather_px, sigmaY=MIX.no_trust_feather_px
        )
        no_trust = np.clip(no_trust, 0.0, 1.0)
    mixed = core * (1.0 - MIX.no_trust_alpha * no_trust[..., None]) + mean * (
        MIX.no_trust_alpha * no_trust[..., None]
    )
    audit["warp_steps"].append(f"no_trust_mix alpha={MIX.no_trust_alpha}")

    mixed = mixed * (1.0 - art[..., None]) + mean * art[..., None]
    audit["warp_steps"].append("ego_overlay")

    if mirrored_proc:
        mixed = mixed[:, ::-1].copy()
        audit["warp_steps"].append(f"proc_mirror_then_unmirror cam={cam}")

    warp_u8 = pwm._f01_u8(mixed)
    audit["warp_source"] = "built_on_the_fly"
    return warp_u8, audit


class V2TrainDataset(Dataset):
    def __init__(self, baked_dirs: list[Path], cfg: V2TrainCfg, ds_cfg: ConsensusConfig):
        self.dirs = [Path(p) for p in baked_dirs]
        self.cfg = cfg
        self.ds_cfg = ds_cfg

    def __len__(self):
        return len(self.dirs)

    def __getitem__(self, idx):
        baked_dir = self.dirs[idx]
        meta = json.loads((baked_dir / "meta.json").read_text(encoding="utf-8"))
        sid = meta["sample_id"]
        cam = meta["camera"]
        src = Path(meta.get("source_dir", self.cfg.dataset_root / sid))
        hw = (self.cfg.image_h, self.cfg.image_w)

        warp_rgb, warp_audit = _load_warp_camera_rgb(self.cfg, self.ds_cfg, sid, cam, src, hw)
        t0 = _load_rgb(src / "input" / "t0" / f"{cam}.jpg", hw)
        t1 = _load_rgb(src / "input" / "t1" / f"{cam}.jpg", hw)
        target_rgb = _load_rgb(src / "target" / f"{cam}.jpg", hw)
        mean_rgb = ((t0.astype(np.float32) + t1.astype(np.float32)) * 0.5).astype(np.uint8)

        trust_path = self.cfg.trust_cache_root / sid / "lidar_trust.npy"
        trust_from_cache = trust_path.is_file()
        if trust_from_cache:
            trust = np.load(trust_path).astype(np.float32)
            if trust.shape != hw:
                trust = cv2.resize(trust, (hw[1], hw[0]), interpolation=cv2.INTER_NEAREST)
        else:
            maps = lidar_rife_blend_maps(
                src, cam, hw, spread_radius_fine=3.0, spread_blur_fine=1.0, zone_min=self.cfg.trust_zone_min
            )
            trust = maps["lidar_trust"].astype(np.float32)
            trust_path.parent.mkdir(parents=True, exist_ok=True)
            np.save(trust_path, trust)
        trust = np.clip(trust, 0.0, 1.0)

        color_steps = []
        warp_ycc = _to_ycrcb01(warp_rgb)
        color_steps.append("warp: RGB -> YCrCb")
        mean_ycc = _to_ycrcb01(mean_rgb)
        color_steps.append("mean_t0_t1: RGB -> YCrCb")
        target_ycc = _to_ycrcb01(target_rgb)
        color_steps.append("target: RGB -> YCrCb")
        color_steps.append("trust: float32, без цветового преобразования")

        mirrored = cam in MIRROR_CAMERAS
        if mirrored:
            warp_ycc, mean_ycc, target_ycc, trust = _flip_h(warp_ycc, mean_ycc, target_ycc, trust)
            color_steps.append(f"mirror_h для модели (cam={cam})")

        x = np.concatenate([warp_ycc, trust[..., None], mean_ycc], axis=-1)
        to_chw = lambda a: torch.from_numpy(a.transpose(2, 0, 1)).contiguous().float()

        loaded = {
            "warp_mix": (self.cfg.warp_mix_root / sid / "warp_mix.jpg").is_file(),
            "consensus_raw": (self.cfg.warps_root / sid / "consensus_raw.npy").is_file(),
            "coverage": (self.cfg.warps_root / sid / "coverage.npy").is_file(),
            "target": (src / "target" / f"{cam}.jpg").is_file(),
            "t0": (src / "input" / "t0" / f"{cam}.jpg").is_file(),
            "t1": (src / "input" / "t1" / f"{cam}.jpg").is_file(),
            "trust_cache": trust_from_cache,
        }

        audit = {
            "sample_id": sid,
            "camera": cam,
            "loaded": loaded,
            "warp_audit": warp_audit,
            "color_steps": color_steps,
            "mirrored_for_model": mirrored,
            "channel_stats": {
                "warp_ycc": {
                    "min": float(warp_ycc.min()),
                    "max": float(warp_ycc.max()),
                    "mean": float(warp_ycc.mean()),
                },
                "trust": {"min": float(trust.min()), "max": float(trust.max()), "mean": float(trust.mean())},
                "mean_ycc": {
                    "min": float(mean_ycc.min()),
                    "max": float(mean_ycc.max()),
                    "mean": float(mean_ycc.mean()),
                },
                "target_ycc": {
                    "min": float(target_ycc.min()),
                    "max": float(target_ycc.max()),
                    "mean": float(target_ycc.mean()),
                },
            },
        }

        return {
            "inputs": to_chw(x),
            "base_init": to_chw(warp_ycc),
            "effective_mask": torch.from_numpy(np.ones(hw, dtype=np.float32))[None],
            "target": to_chw(target_ycc),
            "sample_id": sid,
            "camera": cam,
            "mirrored": mirrored,
            "audit": audit,
        }


## Аудит инпутов

Порядок: **Dataset → Split → `run_input_audit(val_ds)`** (не запускать аудит до Split).

Проверяем: какие файлы загрузились, warp-пайплайн, цветовые шаги, mirror.


In [4]:
def print_input_audit_from_sample(sample: dict):
    """Аудит одного элемента dataset[i] (без DataLoader)."""
    audit = sample["audit"]
    x = sample["inputs"].numpy()
    print("=" * 72)
    print(f"sample_id={sample['sample_id']}  camera={sample['camera']}  mirrored_for_model={sample['mirrored']}")
    print("-" * 72)
    print("Загружено с диска:")
    for k, v in audit["loaded"].items():
        print(f"  [{'OK' if v else 'MISSING'}] {k}")
    print("-" * 72)
    print("Warp:", audit["warp_audit"].get("warp_source"))
    for step in audit["warp_audit"].get("warp_steps", []):
        print(f"  - {step}")
    print("-" * 72)
    print("Цветовые преобразования:")
    for step in audit["color_steps"]:
        print(f"  - {step}")
    print("-" * 72)
    print("Статистика каналов:")
    for name, st in audit["channel_stats"].items():
        print(f"  {name:10s} min={st['min']:.4f} max={st['max']:.4f} mean={st['mean']:.4f}")
    ch_names = ["warp_Y", "warp_Cr", "warp_Cb", "trust", "mean_Y", "mean_Cr", "mean_Cb"]
    print("-" * 72)
    print("Тензор inputs [7,H,W]:")
    for ci, nm in enumerate(ch_names):
        ch = x[ci]
        print(f"  ch{ci} {nm:8s} min={ch.min():.4f} max={ch.max():.4f} mean={ch.mean():.4f}")
    warp_rgb = _ycrcb01_to_rgb01(x[:3].transpose(1, 2, 0))
    mean_rgb = _ycrcb01_to_rgb01(x[4:7].transpose(1, 2, 0))
    print("-" * 72)
    print("Round-trip YCrCb->RGB:")
    print(f"  warp_rgb  min={warp_rgb.min():.4f} max={warp_rgb.max():.4f}")
    print(f"  mean_rgb  min={mean_rgb.min():.4f} max={mean_rgb.max():.4f}")
    print("=" * 72)


def print_input_audit(batch, sample_idx: int = 0):
    if "audit" not in batch:
        raise KeyError(
            "В batch нет 'audit'. Сначала выполните ячейки Dataset + Split, "
            "убедитесь что val_ds = V2TrainDataset и DataLoader(..., collate_fn=collate_v2_batch)."
        )
    i = int(np.clip(sample_idx, 0, batch["inputs"].shape[0] - 1))
    audit = batch["audit"][i]

    print("=" * 72)
    print(
        f"sample_id={batch['sample_id'][i]}  camera={batch['camera'][i]}  "
        f"mirrored_for_model={batch['mirrored'][i]}"
    )
    print("-" * 72)
    print("Загружено с диска:")
    for k, v in audit["loaded"].items():
        mark = "OK" if v else "MISSING"
        print(f"  [{mark}] {k}")

    print("-" * 72)
    print("Warp:", audit["warp_audit"].get("warp_source"))
    for step in audit["warp_audit"].get("warp_steps", []):
        print(f"  - {step}")

    print("-" * 72)
    print("Цветовые преобразования:")
    for step in audit["color_steps"]:
        print(f"  - {step}")

    print("-" * 72)
    print("Статистика каналов (после всех преобразований):")
    for name, st in audit["channel_stats"].items():
        print(f"  {name:10s} min={st['min']:.4f} max={st['max']:.4f} mean={st['mean']:.4f}")

    x = batch["inputs"][i].numpy()
    ch_names = ["warp_Y", "warp_Cr", "warp_Cb", "trust", "mean_Y", "mean_Cr", "mean_Cb"]
    print("-" * 72)
    print("Тензор inputs [7,H,W]:")
    for ci, nm in enumerate(ch_names):
        ch = x[ci]
        print(f"  ch{ci} {nm:8s} min={ch.min():.4f} max={ch.max():.4f} mean={ch.mean():.4f}")

    warp_rgb = _ycrcb01_to_rgb01(x[:3].transpose(1, 2, 0))
    mean_rgb = _ycrcb01_to_rgb01(x[4:7].transpose(1, 2, 0))
    print("-" * 72)
    print("Round-trip YCrCb->RGB:")
    print(f"  warp_rgb  min={warp_rgb.min():.4f} max={warp_rgb.max():.4f}")
    print(f"  mean_rgb  min={mean_rgb.min():.4f} max={mean_rgb.max():.4f}")
    print("=" * 72)


def run_input_audit(dataset, n_samples: int = 4, batch_size: int = 8):
    """Запускать после ячейки Split (когда val_ds уже создан)."""
    probe = dataset[0]
    if "audit" not in probe:
        raise RuntimeError("dataset не V2TrainDataset — перезапустите ячейку Dataset.")

    loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=False, num_workers=0, collate_fn=collate_v2_batch
    )
    batch = next(iter(loader))
    n = min(n_samples, len(batch["audit"]))
    print(f"Аудит батча: {n} сэмплов, keys={list(batch.keys())}\n")
    for ai in range(n):
        print_input_audit(batch, sample_idx=ai)
        print()


# Вызов — в следующей ячейке Split (после создания val_ds):
# run_input_audit(val_ds, n_samples=4)


In [5]:
seed_everything(CFG.seed)

all_samples = discover_samples(DS_CFG)
print(f"discover_samples (все камеры): {len(all_samples)}")

ready = []
for bdir in all_samples:
    meta = json.loads((bdir / "meta.json").read_text(encoding="utf-8"))
    sid = meta["sample_id"]
    if (CFG.trust_cache_root / sid / "lidar_trust.npy").is_file():
        ready.append(bdir)

print(f"ready_with_trust: {len(ready)}")

from collections import Counter

cams = Counter(json.loads((p / "meta.json").read_text())["camera"] for p in ready)
print("камеры:", dict(sorted(cams.items())))

rng = np.random.RandomState(CFG.seed)
idx = np.arange(len(ready))
rng.shuffle(idx)
if CFG.max_samples and len(idx) > CFG.max_samples:
    idx = idx[: CFG.max_samples]

chosen = [ready[i] for i in idx]
n_val = max(1, int(len(chosen) * CFG.val_fraction))
val_dirs = chosen[:n_val]
train_dirs = chosen[n_val:]
print(f"using samples={len(chosen)} train={len(train_dirs)} val={len(val_dirs)}")

train_ds = V2TrainDataset(train_dirs, CFG, DS_CFG)
val_ds = V2TrainDataset(val_dirs, CFG, DS_CFG)
train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True, num_workers=CFG.num_workers, collate_fn=collate_v2_batch)
val_loader = DataLoader(
    val_ds, batch_size=CFG.val_batch_size, shuffle=False, num_workers=CFG.num_workers, collate_fn=collate_v2_batch
)

b = next(iter(train_loader))
print("inputs:", tuple(b["inputs"].shape), "target:", tuple(b["target"].shape))
print("batch keys:", list(b.keys()))

# Аудит val (нужны ячейки Dataset + эта Split)
run_input_audit(val_ds, n_samples=4)


discover_samples (все камеры): 1278


ready_with_trust: 1278
камеры: {'front': 197, 'left_bwd': 214, 'left_fwd': 206, 'rear': 252, 'right_bwd': 199, 'right_fwd': 210}
using samples=1278 train=1074 val=204


inputs: (12, 7, 544, 1024) target: (12, 3, 544, 1024)
batch keys: ['inputs', 'base_init', 'effective_mask', 'target', 'sample_id', 'camera', 'mirrored', 'audit']


Аудит батча: 4 сэмплов, keys=['inputs', 'base_init', 'effective_mask', 'target', 'sample_id', 'camera', 'mirrored', 'audit']

sample_id=2025-11-02_18_06_30_18_09_54_susana_1762097049299880000__001  camera=front  mirrored_for_model=False
------------------------------------------------------------------------
Загружено с диска:
  [OK] warp_mix
  [OK] consensus_raw
  [OK] coverage
  [OK] target
  [OK] t0
  [OK] t1
  [OK] trust_cache
------------------------------------------------------------------------
Warp: side_warp_mix_v1/warp_mix.jpg
  - load_precomputed_camera_space
------------------------------------------------------------------------
Цветовые преобразования:
  - warp: RGB -> YCrCb
  - mean_t0_t1: RGB -> YCrCb
  - target: RGB -> YCrCb
  - trust: float32, без цветового преобразования
------------------------------------------------------------------------
Статистика каналов (после всех преобразований):
  warp_ycc   min=0.0000 max=0.9882 mean=0.4476
  trust      min=0.0000 max=1.

## Архитектура: ConsensusUNet

Модель из `consensus_kit.py` — **U-Net с Partial Convolution** (как в inpainting):

- **Энкодер**: 4 уровня `PartialDoubleConv` + pooling; маска `eff_mask` обновляется на каждом слое (дыры не «размазываются»).
- **Боттleneck**: обычный `ConvBlock`.
- **Декодер**: skip + **маска** на каждом уровне (`cat([up, skip, mask])`).
- **Выход**: `pred = clamp(base_init + residual, 0, 1)` — учится **остаток к warp**, не картинка с нуля.

V2: `in_ch=7`, `predict_confidence=False`, `base_init` = warp в YCrCb.

**Почему подходит задаче:** warp уже близок к target, нужна локальная доработка (дыры, no-trust, цвет); partial conv + residual — стандартный выбор для completion/refinement.

**Ограничения / что можно улучшить позже:**
- Один U-Net на все камеры — можно добавить **camera embedding** (1–4 канала или FiLM).
- При 1024×544 receptive field боттleneck ограничен — иногда помогает **attention** в bottleneck (Restormer/NAFNet-блок).
- Если bottleneck по качеству — отдельная голова только на **Cr/Cb** (Y почти из warp).
- Для сравнения baseline: тот же U-Net в RGB vs YCrCb (у вас YCrCb уже выигрывает на 100 сэмплах).

Сейчас **менять архитектуру не обязательно** — сначала добить данные/препроцесс и full-train eval; смена сети имеет смысл, если val PSNR упирается при хороших инпутах.

In [6]:
# === Stage-1 Train (опционально) ===
RUN_STAGE1_TRAIN = False  # True = заново учить Stage-1 YCrCb

if "device" not in globals():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if not RUN_STAGE1_TRAIN:
    print("SKIP Stage-1 train. Используй ячейку «Load Stage-1», затем Stage-2.")
else:
    print("device:", device)
    if device.type == "cuda":
        torch.cuda.empty_cache()
        _props = torch.cuda.get_device_properties(0)
        print(f"GPU: {_props.name}, VRAM ~{_props.total_memory / 1024**3:.1f} GB")
    print(f"train batch_size={CFG.batch_size}, val batch_size={CFG.val_batch_size}")

    model = ConsensusUNet(
        in_ch=CFG.in_channels,
        base=CFG.base_channels,
        predict_confidence=False,
    ).to(device)
    print(f"base_channels={CFG.base_channels}")

    opt = torch.optim.AdamW(model.parameters(), lr=CFG.lr)
    criterion = torch.nn.MSELoss()
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    def run_epoch(loader, train: bool):
        model.train(train)
        losses = []
        for batch in loader:
            inp = batch["inputs"].to(device, non_blocking=True)
            base = batch["base_init"].to(device, non_blocking=True)
            msk = batch["effective_mask"].to(device, non_blocking=True)
            tgt = batch["target"].to(device, non_blocking=True)
            with torch.set_grad_enabled(train):
                with torch.amp.autocast("cuda", enabled=use_amp):
                    out = model(inp, msk, base)["pred"].float()
                    loss = criterion(out, tgt)
                if train:
                    opt.zero_grad(set_to_none=True)
                    scaler.scale(loss).backward()
                    scaler.step(opt)
                    scaler.update()
            losses.append(float(loss.detach().cpu()))
            del inp, base, msk, tgt, out, loss
        if device.type == "cuda":
            torch.cuda.empty_cache()
        return float(np.mean(losses)) if losses else 0.0

    best_val = 1e9
    save_dir = Path("artifacts/checkpoints/consensus_v2_all")
    save_dir.mkdir(parents=True, exist_ok=True)
    ckpt_best = save_dir / "consensus_v2_all_best.pt"

    for ep in range(1, CFG.epochs + 1):
        tr = run_epoch(train_loader, train=True)
        va = run_epoch(val_loader, train=False)
        print(f"epoch {ep:02d}: train_mse={tr:.6f}  val_mse={va:.6f}")
        if va < best_val:
            best_val = va
            torch.save({
                "epoch": ep,
                "model": model.state_dict(),
                "opt": opt.state_dict(),
                "cfg": vars(CFG),
                "best_val_mse": best_val,
                "color_space": "YCrCb",
                "inputs": ["warp_ycc(3)", "lidar_trust(1)", "mean_t0t1_ycc(3)"],
                "target": "target_ycc(3)",
                "loss": "MSE",
            }, ckpt_best)
            print("  saved best ->", ckpt_best)
    print("done. best_val_mse=", best_val)


SKIP Stage-1 train. Используй ячейку «Load Stage-1», затем Stage-2.


## Debug: внешние скрипты (опционально)

**Закомментированные строки — не обязательны** для основного пайплайна в ноутбуке:

| Строка | Запускать? |
|--------|------------|
| `# run_input_audit(...)` в ячейке аудита | Нет — аудит уже в Split |
| `# m_ycc = train_and_eval(...)` | Только если нужен **отдельный** прогон через CLI на 100 сэмплах (дублирует train-ячейку) |
| `# m_rgb = train_and_eval(...)` | То же для RGB baseline |
| `# compare_saved_metrics()` | Только после того, как сохранили `metrics.json` от eval |
| `# visualize_v2_sample(...)` | **Да**, после обучения — раскомментировать для картинок |

Основной путь: **CFG → Dataset → Split → Train → visualize**.

In [7]:
import json
import subprocess
from pathlib import Path

REPO = Path(r"C:/Users/adel/Documents/GitHub/YA-")
TRAIN_SCRIPT = REPO / "scripts/training/train_v2_ycrcb_side.py"
EVAL_SCRIPT = REPO / "scripts/training/eval_v2_ycrcb_side.py"


def run_cmd(cmd: list[str]):
    print("$", " ".join(cmd))
    return subprocess.run(cmd, cwd=str(REPO), check=True)


def train_and_eval(color_space: str, limit=100, epochs=3, batch_size=10, device="cuda", save_n=8):
    ckpt = REPO / f"artifacts/checkpoints/consensus_side_v2/consensus_v2_side_100_{color_space}_best.pt"
    out_dir = REPO / f"artifacts/preview/consensus_v2_side_eval_{color_space}"

    run_cmd([
        "python", str(TRAIN_SCRIPT),
        "--limit", str(limit),
        "--epochs", str(epochs),
        "--batch-size", str(batch_size),
        "--device", device,
        "--color-space", color_space,
    ])

    run_cmd([
        "python", str(EVAL_SCRIPT),
        "--ckpt", str(ckpt),
        "--color-space", color_space,
        "--limit", str(limit),
        "--save-n", str(save_n),
        "--batch-size", "8",
        "--device", device,
        "--out-dir", str(out_dir),
    ])

    metrics_path = out_dir / "metrics.json"
    if metrics_path.is_file():
        metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
        print(f"\n[{color_space}] MSE_RGB={metrics['metrics']['mse_rgb_mean']:.6f}  PSNR_RGB={metrics['metrics']['psnr_rgb_mean']:.3f} dB")
        return metrics
    print("metrics.json not found:", metrics_path)
    return None


# Примеры ручного запуска:
# m_ycc = train_and_eval("ycrcb", limit=100, epochs=3, batch_size=10, device="cuda")
# m_rgb = train_and_eval("rgb", limit=100, epochs=3, batch_size=10, device="cuda")


def compare_saved_metrics():
    paths = {
        "ycrcb": REPO / "artifacts/preview/consensus_v2_side_eval_ycrcb/metrics.json",
        "rgb": REPO / "artifacts/preview/consensus_v2_side_eval_rgb/metrics.json",
    }
    rows = []
    for k, p in paths.items():
        if p.is_file():
            m = json.loads(p.read_text(encoding="utf-8"))
            rows.append((k, m["metrics"]["mse_rgb_mean"], m["metrics"]["psnr_rgb_mean"], m["metrics"]["mse_ycc_mean"], m["metrics"]["psnr_ycc_mean"]))

    if not rows:
        print("Нет metrics.json для сравнения")
        return

    print("color_space | mse_rgb | psnr_rgb | mse_ycc | psnr_ycc")
    for r in rows:
        print(f"{r[0]:10s} | {r[1]:.6f} | {r[2]:.3f} | {r[3]:.6f} | {r[4]:.3f}")


# compare_saved_metrics()

## Визуализация: инпуты, аутпут (YCrCb -> RGB), карта ошибок

Эта секция показывает:
- входные данные модели (`warp_ycc`, `lidar_trust`, `mean_t0t1_ycc`),
- предсказание в YCrCb (до конвертации в RGB),
- предсказание в RGB и target RGB,
- карты ошибок (`|pred-target|`) в YCrCb и RGB.

In [8]:
import matplotlib.pyplot as plt


def ycc01_to_rgb01(ycc01: np.ndarray) -> np.ndarray:
    u8 = np.clip(ycc01 * 255.0, 0, 255).astype(np.uint8)
    rgb = cv2.cvtColor(u8, cv2.COLOR_YCrCb2RGB).astype(np.float32) / 255.0
    return np.clip(rgb, 0.0, 1.0)


@torch.no_grad()
def visualize_v2_sample(model, loader, device, sample_idx: int = 0):
    model.eval()
    batch = next(iter(loader))

    inp = batch["inputs"].to(device)
    base = batch["base_init"].to(device)
    msk = batch["effective_mask"].to(device)
    tgt = batch["target"].to(device)

    pred = model(inp, msk, base)["pred"].float().clamp(0, 1)

    i = int(np.clip(sample_idx, 0, pred.shape[0] - 1))

    x = inp[i].detach().cpu().numpy()               # [7,H,W]
    p_ycc = pred[i].detach().cpu().permute(1, 2, 0).numpy()
    t_ycc = tgt[i].detach().cpu().permute(1, 2, 0).numpy()

    warp_ycc = np.transpose(x[0:3], (1, 2, 0))
    trust = x[3]
    mean_ycc = np.transpose(x[4:7], (1, 2, 0))

    p_rgb = ycc01_to_rgb01(p_ycc)
    t_rgb = ycc01_to_rgb01(t_ycc)
    warp_rgb = ycc01_to_rgb01(warp_ycc)
    mean_rgb = ycc01_to_rgb01(mean_ycc)

    err_ycc = np.abs(p_ycc - t_ycc).mean(axis=2)
    err_rgb = np.abs(p_rgb - t_rgb).mean(axis=2)

    mse_ycc = float(np.mean((p_ycc - t_ycc) ** 2))
    mse_rgb = float(np.mean((p_rgb - t_rgb) ** 2))
    psnr_ycc = 99.0 if mse_ycc <= 1e-12 else float(10 * np.log10(1.0 / mse_ycc))
    psnr_rgb = 99.0 if mse_rgb <= 1e-12 else float(10 * np.log10(1.0 / mse_rgb))

    fig, ax = plt.subplots(3, 4, figsize=(16, 10))

    ax[0, 0].imshow(warp_rgb); ax[0, 0].set_title("Input warp (RGB view)")
    ax[0, 1].imshow(trust, cmap="viridis", vmin=0, vmax=1); ax[0, 1].set_title("Input lidar_trust")
    ax[0, 2].imshow(mean_rgb); ax[0, 2].set_title("Input mean(t0,t1) (RGB view)")
    ax[0, 3].imshow(np.abs(warp_rgb - mean_rgb).mean(axis=2), cmap="magma"); ax[0, 3].set_title("|warp-mean|")

    ax[1, 0].imshow(p_ycc[..., 0], cmap="gray", vmin=0, vmax=1); ax[1, 0].set_title("Pred Y (YCrCb)")
    ax[1, 1].imshow(p_ycc[..., 1], cmap="coolwarm", vmin=0, vmax=1); ax[1, 1].set_title("Pred Cr")
    ax[1, 2].imshow(p_ycc[..., 2], cmap="coolwarm", vmin=0, vmax=1); ax[1, 2].set_title("Pred Cb")
    ax[1, 3].imshow(err_ycc, cmap="inferno"); ax[1, 3].set_title(f"Err map YCrCb mean\nMSE={mse_ycc:.5f} PSNR={psnr_ycc:.2f}dB")

    ax[2, 0].imshow(p_rgb); ax[2, 0].set_title("Pred RGB (after YCrCb->RGB)")
    ax[2, 1].imshow(t_rgb); ax[2, 1].set_title("Target RGB")
    ax[2, 2].imshow(np.clip(np.abs(p_rgb - t_rgb) * 4.0, 0, 1)); ax[2, 2].set_title("|Pred-Target| RGB x4")
    ax[2, 3].imshow(err_rgb, cmap="inferno"); ax[2, 3].set_title(f"Err map RGB mean\nMSE={mse_rgb:.5f} PSNR={psnr_rgb:.2f}dB")

    for r in range(3):
        for c in range(4):
            ax[r, c].axis("off")

    sid = batch.get("sample_id", ["?"])[i]
    cam = batch.get("camera", ["?"])[i]
    mir = batch.get("mirrored", [False])[i]
    fig.suptitle(f"sample={sid}  cam={cam}  mirrored={mir}", fontsize=12)
    plt.tight_layout()
    plt.show()


# Пример:
visualize_v2_sample(model, val_loader, device, sample_idx=0)

NameError: name 'model' is not defined

In [9]:
import numpy as np
import cv2
import torch
def ycc01_to_rgb01(ycc):
    u8 = np.clip(ycc * 255.0, 0, 255).astype(np.uint8)
    rgb = cv2.cvtColor(u8, cv2.COLOR_YCrCb2RGB).astype(np.float32) / 255.0
    return np.clip(rgb, 0.0, 1.0)
def psnr01(a, b):
    mse = float(np.mean((a - b) ** 2))
    return 99.0 if mse <= 1e-12 else float(10.0 * np.log10(1.0 / mse))
model.eval()
mse_rgb_all, psnr_rgb_all = [], []
with torch.no_grad():
    for batch in val_loader:
        inp = batch["inputs"].to(device)
        base = batch["base_init"].to(device)
        msk = batch["effective_mask"].to(device)
        tgt = batch["target"].to(device)
        pred = model(inp, msk, base)["pred"].float().clamp(0, 1)
        for i in range(pred.shape[0]):
            p_ycc = pred[i].permute(1,2,0).cpu().numpy()
            t_ycc = tgt[i].permute(1,2,0).cpu().numpy()
            p_rgb = ycc01_to_rgb01(p_ycc)
            t_rgb = ycc01_to_rgb01(t_ycc)
            mse = float(np.mean((p_rgb - t_rgb) ** 2))
            mse_rgb_all.append(mse)
            psnr_rgb_all.append(psnr01(p_rgb, t_rgb))
print(f"VAL RGB MSE:  {np.mean(mse_rgb_all):.6f}")
print(f"VAL RGB PSNR: {np.mean(psnr_rgb_all):.3f} dB")
print(f"VAL samples:  {len(mse_rgb_all)}")

NameError: name 'model' is not defined

In [ ]:
import numpy as np
import cv2
import torch
from pathlib import Path
import json
from PIL import Image

RIFE_ROOT = Path(r"C:/Users/adel/Downloads/cv_dataset/rife_predictions_v5/train")
alphas = np.linspace(0.0, 1.0, 11)  # out = a*pred + (1-a)*rife

def ycc01_to_rgb01(ycc01):
    u8 = np.clip(ycc01 * 255.0, 0, 255).astype(np.uint8)
    return cv2.cvtColor(u8, cv2.COLOR_YCrCb2RGB).astype(np.float32) / 255.0

def psnr01(a, b):
    mse = float(np.mean((a - b) ** 2))
    if mse <= 1e-12:
        return 99.0
    return float(10.0 * np.log10(1.0 / mse))

def load_rgb01(path: Path, hw):
    arr = np.array(Image.open(path).convert("RGB"), dtype=np.uint8)
    if arr.shape[:2] != hw:
        arr = cv2.resize(arr, (hw[1], hw[0]), interpolation=cv2.INTER_LINEAR)
    return arr.astype(np.float32) / 255.0

def find_rife_path_by_sid(sid: str) -> Path | None:
    # основной формат
    cands = [
        RIFE_ROOT / f"{sid}.jpg",
        RIFE_ROOT / f"{sid}.png",
        # запасные
        RIFE_ROOT / sid / "pred.jpg",
        RIFE_ROOT / sid / "pred.png",
        RIFE_ROOT / sid / f"{sid}.jpg",
        RIFE_ROOT / sid / f"{sid}.png",
    ]
    for p in cands:
        if p.is_file():
            return p
    return None

model.eval()
stats = {float(a): {"mse": [], "psnr": []} for a in alphas}
missing_rife = 0
used = 0
global_idx = 0
hw = (CFG.image_h, CFG.image_w)

with torch.no_grad():
    for batch in val_loader:
        inp = batch["inputs"].to(device)
        base = batch["base_init"].to(device)
        msk = batch["effective_mask"].to(device)
        tgt = batch["target"].to(device)

        pred = model(inp, msk, base)["pred"].float().clamp(0, 1)

        for i in range(pred.shape[0]):
            baked_dir = val_ds.dirs[global_idx]
            meta = json.loads((Path(baked_dir) / "meta.json").read_text(encoding="utf-8"))
            sid = meta["sample_id"]

            rife_path = find_rife_path_by_sid(sid)
            if rife_path is None:
                missing_rife += 1
                global_idx += 1
                continue

            p_ycc = pred[i].permute(1, 2, 0).cpu().numpy().astype(np.float32)
            t_ycc = tgt[i].permute(1, 2, 0).cpu().numpy().astype(np.float32)

            pred_rgb = ycc01_to_rgb01(p_ycc)
            tgt_rgb = ycc01_to_rgb01(t_ycc)
            rife_rgb = load_rgb01(rife_path, hw)

            # align orientation to model input orientation
            mirrored = bool(batch["mirrored"][i]) if "mirrored" in batch else False
            if mirrored:
                rife_rgb = rife_rgb[:, ::-1].copy()

            for a in alphas:
                mix = np.clip(float(a) * pred_rgb + (1.0 - float(a)) * rife_rgb, 0.0, 1.0)
                mse = float(np.mean((mix - tgt_rgb) ** 2))
                stats[float(a)]["mse"].append(mse)
                stats[float(a)]["psnr"].append(psnr01(mix, tgt_rgb))

            used += 1
            global_idx += 1

print(f"used samples: {used}, missing_rife: {missing_rife}")

rows = []
for a in alphas:
    vals = stats[float(a)]["mse"]
    if len(vals) == 0:
        continue
    rows.append((
        float(a),
        float(np.mean(stats[float(a)]["mse"])),
        float(np.mean(stats[float(a)]["psnr"]))
    ))

if not rows:
    print("Не найдено ни одного совпадения RIFE. Проверь путь RIFE_ROOT и имена файлов.")
else:
    rows = sorted(rows, key=lambda x: x[1])  # by mse
    print("\nTop-5 by RGB MSE:")
    for a, mse, psnr in rows[:5]:
        print(f"alpha={a:.2f}  mse_rgb={mse:.6f}  psnr_rgb={psnr:.3f} dB")

    best_a, best_mse, best_psnr = rows[0]
    print(f"\nBEST: alpha={best_a:.2f}, mse_rgb={best_mse:.6f}, psnr_rgb={best_psnr:.3f} dB")

## Stage 2 only (перезапуск без обучения Stage-1)

**Порядок:** Imports → CFG → Dataset → Split → **Load Stage-1** → Export pseudo → Stage-2 train.

**Не запускай** ячейку `Stage-1 Train` (там `RUN_STAGE1_TRAIN = False`).

**Вход Stage-2 (7 ch, RGB):** `pred_rgb` + `lidar_trust` + `rife_rgb` → target RGB (MSE).


In [10]:
# === Load Stage-1 (без переобучения) ===
STAGE1_CKPT = REPO / "notebooks/training/artifacts/checkpoints/consensus_v2_all/consensus_v2_all_best.pt"
if not STAGE1_CKPT.is_file():
    STAGE1_CKPT = Path("artifacts/checkpoints/consensus_v2_all/consensus_v2_all_best.pt")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("Stage-1 ckpt:", STAGE1_CKPT, "exists:", STAGE1_CKPT.is_file())

st1 = torch.load(STAGE1_CKPT, map_location=device, weights_only=False)
base_ch = int(st1.get("cfg", {}).get("base_channels", CFG.base_channels))

model = ConsensusUNet(in_ch=CFG.in_channels, base=base_ch, predict_confidence=False).to(device)
model.load_state_dict(st1["model"], strict=True)
model.eval()
print("Stage-1 loaded. best_val_mse=", st1.get("best_val_mse"))


device: cuda
Stage-1 ckpt: C:\Users\adel\Documents\GitHub\YA-\notebooks\training\artifacts\checkpoints\consensus_v2_all\consensus_v2_all_best.pt exists: True
Stage-1 loaded. best_val_mse= 0.0025435057194793925


In [11]:
PSEUDO_ROOT = CV_ROOT / "pseudo_v2_train_rgb"
FORCE_REEXPORT_PSEUDO = False  # True = заново прогнать stage-1 на train

@dataclass
class Stage2Cfg:
    image_h: int = CFG.image_h
    image_w: int = CFG.image_w
    batch_size: int = 12
    val_batch_size: int = 12
    num_workers: int = 0
    lr: float = 1e-4
    epochs: int = 40
    in_channels: int = 7   # pred_rgb(3) + lidar_trust(1) + rife_rgb(3)
    base_channels: int = 32

S2 = Stage2Cfg()
print(S2)


Stage2Cfg(image_h=544, image_w=1024, batch_size=12, val_batch_size=12, num_workers=0, lr=0.0001, epochs=40, in_channels=7, base_channels=32)


In [12]:
def ycc01_to_rgb01(ycc01: np.ndarray) -> np.ndarray:
    u8 = np.clip(ycc01 * 255.0, 0, 255).astype(np.uint8)
    return cv2.cvtColor(u8, cv2.COLOR_YCrCb2RGB).astype(np.float32) / 255.0


def _pseudo_ready(sid: str, root: Path) -> bool:
    d = root / sid
    return (d / "pred_rgb.npy").is_file() and (d / "target_rgb.npy").is_file()


@torch.no_grad()
def export_train_preds_rgb(model, dataset, out_root: Path, batch_size=8, device=device, force=False):
    out_root.mkdir(parents=True, exist_ok=True)
    loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=False, num_workers=0, collate_fn=collate_v2_batch
    )
    model.eval()
    n_new, n_skip = 0, 0

    for batch in loader:
        inp = batch["inputs"].to(device)
        base = batch["base_init"].to(device)
        msk = batch["effective_mask"].to(device)
        pred = model(inp, msk, base)["pred"].float().clamp(0, 1).cpu().numpy()

        for i in range(pred.shape[0]):
            sid = batch["sample_id"][i]
            cam = batch["camera"][i]
            mirrored = bool(batch["mirrored"][i])

            if not force and _pseudo_ready(sid, out_root):
                n_skip += 1
                continue

            pred_rgb = ycc01_to_rgb01(np.transpose(pred[i], (1, 2, 0)))
            tgt_rgb = ycc01_to_rgb01(np.transpose(batch["target"][i].numpy(), (1, 2, 0)))
            if mirrored:
                pred_rgb = pred_rgb[:, ::-1].copy()
                tgt_rgb = tgt_rgb[:, ::-1].copy()

            d = out_root / sid
            d.mkdir(parents=True, exist_ok=True)
            np.save(d / "pred_rgb.npy", pred_rgb.astype(np.float32))
            np.save(d / "target_rgb.npy", tgt_rgb.astype(np.float32))
            (d / "meta.json").write_text(
                json.dumps({"sample_id": sid, "camera": cam}, ensure_ascii=False), encoding="utf-8"
            )
            n_new += 1

    print(f"pseudo RGB: new={n_new} skip={n_skip} -> {out_root}")


all_dirs = train_dirs + val_dirs
export_train_preds_rgb(
    model,
    V2TrainDataset(all_dirs, CFG, DS_CFG),
    PSEUDO_ROOT,
    batch_size=CFG.val_batch_size,
    device=device,
    force=FORCE_REEXPORT_PSEUDO,
)


pseudo RGB: new=0 skip=1278 -> C:\Users\adel\Downloads\cv_dataset\pseudo_v2_train_rgb


In [13]:
def load_rgb_u8(path: Path, hw):
    arr = np.array(Image.open(path).convert("RGB"), dtype=np.uint8)
    if arr.shape[:2] != hw:
        arr = cv2.resize(arr, (hw[1], hw[0]), interpolation=cv2.INTER_LINEAR)
    return arr


def load_rife_rgb01(cfg, sid: str, hw) -> np.ndarray:
    p = cfg.rife_root / f"{sid}.jpg"
    if not p.is_file():
        p = cfg.rife_root / f"{sid}.png"
    if not p.is_file():
        raise FileNotFoundError(f"RIFE not found: {sid}")
    return load_rgb_u8(p, hw).astype(np.float32) / 255.0


class Stage2RGBDataset(Dataset):
    """RGB: pred(stage1) + trust + RIFE -> target."""

    def __init__(self, baked_dirs, cfg, s2cfg, pseudo_root: Path):
        self.dirs = [Path(x) for x in baked_dirs]
        self.cfg = cfg
        self.s2 = s2cfg
        self.pseudo_root = pseudo_root

    def __len__(self):
        return len(self.dirs)

    def __getitem__(self, idx):
        baked_dir = self.dirs[idx]
        meta = json.loads((baked_dir / "meta.json").read_text(encoding="utf-8"))
        sid, cam = meta["sample_id"], meta["camera"]
        hw = (self.s2.image_h, self.s2.image_w)

        pred_rgb = np.load(self.pseudo_root / sid / "pred_rgb.npy").astype(np.float32)
        target_rgb = np.load(self.pseudo_root / sid / "target_rgb.npy").astype(np.float32)
        trust = np.load(self.cfg.trust_cache_root / sid / "lidar_trust.npy").astype(np.float32)
        if trust.shape != hw:
            trust = cv2.resize(trust, (hw[1], hw[0]), interpolation=cv2.INTER_NEAREST)
        trust = np.clip(trust, 0.0, 1.0)
        rife_rgb = load_rife_rgb01(self.cfg, sid, hw)

        if cam in MIRROR_CAMERAS:
            pred_rgb, target_rgb, trust, rife_rgb = _flip_h(pred_rgb, target_rgb, trust, rife_rgb)

        x = np.concatenate([pred_rgb, trust[..., None], rife_rgb], axis=-1)
        to_chw = lambda a: torch.from_numpy(a.transpose(2, 0, 1)).contiguous().float()
        to_1chw = lambda a: torch.from_numpy(a)[None].contiguous().float()

        return {
            "inputs": to_chw(x),
            "base_init": to_chw(pred_rgb),
            "effective_mask": to_1chw(np.ones(hw, dtype=np.float32)),
            "target": to_chw(target_rgb),
            "sample_id": sid,
            "camera": cam,
        }


In [ ]:
s2_train_ds = Stage2RGBDataset(train_dirs, CFG, S2, PSEUDO_ROOT)
s2_val_ds = Stage2RGBDataset(val_dirs, CFG, S2, PSEUDO_ROOT)

s2_train_loader = DataLoader(s2_train_ds, batch_size=S2.batch_size, shuffle=True, num_workers=S2.num_workers)
s2_val_loader = DataLoader(s2_val_ds, batch_size=S2.val_batch_size, shuffle=False, num_workers=S2.num_workers)

s2_model = ConsensusUNet(in_ch=S2.in_channels, base=S2.base_channels, predict_confidence=False).to(device)
s2_opt = torch.optim.AdamW(s2_model.parameters(), lr=S2.lr)
s2_criterion = torch.nn.MSELoss()
s2_scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))


def run_s2_epoch(loader, train=True):
    s2_model.train(train)
    losses = []
    for b in loader:
        inp = b["inputs"].to(device, non_blocking=True)
        base = b["base_init"].to(device, non_blocking=True)
        msk = b["effective_mask"].to(device, non_blocking=True)
        tgt = b["target"].to(device, non_blocking=True)
        with torch.set_grad_enabled(train):
            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                out = s2_model(inp, msk, base)["pred"].float()
                loss = s2_criterion(out, tgt)
            if train:
                s2_opt.zero_grad(set_to_none=True)
                s2_scaler.scale(loss).backward()
                s2_scaler.step(s2_opt)
                s2_scaler.update()
        losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses)) if losses else 0.0


best_s2 = 1e9
s2_ckpt = REPO / "notebooks/training/artifacts/checkpoints/consensus_v2_all/stage2_rgb_pred_trust_rife_best.pt"
s2_ckpt.parent.mkdir(parents=True, exist_ok=True)

for ep in range(1, S2.epochs + 1):
    tr = run_s2_epoch(s2_train_loader, True)
    va = run_s2_epoch(s2_val_loader, False)
    print(f"[S2] ep{ep:02d} train_mse_rgb={tr:.6f} val_mse_rgb={va:.6f}")
    if va < best_s2:
        best_s2 = va
        torch.save(
            {
                "model": s2_model.state_dict(),
                "best_val_mse_rgb": best_s2,
                "cfg": vars(S2),
                "inputs": ["pred_rgb(3)", "lidar_trust(1)", "rife_rgb(3)"],
                "color_space": "RGB",
            },
            s2_ckpt,
        )
        print("  saved ->", s2_ckpt)

print("done best_val_mse_rgb=", best_s2)


[S2] ep01 train_mse_rgb=0.003565 val_mse_rgb=0.008660
  saved -> C:\Users\adel\Documents\GitHub\YA-\notebooks\training\artifacts\checkpoints\consensus_v2_all\stage2_rgb_pred_trust_rife_best.pt


[S2] ep02 train_mse_rgb=0.003557 val_mse_rgb=0.008636
  saved -> C:\Users\adel\Documents\GitHub\YA-\notebooks\training\artifacts\checkpoints\consensus_v2_all\stage2_rgb_pred_trust_rife_best.pt


[S2] ep03 train_mse_rgb=0.003559 val_mse_rgb=0.008573
  saved -> C:\Users\adel\Documents\GitHub\YA-\notebooks\training\artifacts\checkpoints\consensus_v2_all\stage2_rgb_pred_trust_rife_best.pt


[S2] ep04 train_mse_rgb=0.003555 val_mse_rgb=0.008557
  saved -> C:\Users\adel\Documents\GitHub\YA-\notebooks\training\artifacts\checkpoints\consensus_v2_all\stage2_rgb_pred_trust_rife_best.pt


[S2] ep05 train_mse_rgb=0.003554 val_mse_rgb=0.008573


...

[S2] ep14 train_mse_rgb=0.003536 val_mse_rgb=0.008552


[S2] ep15 train_mse_rgb=0.003532 val_mse_rgb=0.008490
  saved -> C:\Users\adel\Documents\GitHub\YA-\notebooks\training\artifacts\checkpoints\consensus_v2_all\stage2_rgb_pred_trust_rife_best.pt


[S2] ep16 train_mse_rgb=0.003522 val_mse_rgb=0.008533


[S2] ep17 train_mse_rgb=0.003519 val_mse_rgb=0.008547


[S2] ep18 train_mse_rgb=0.003512 val_mse_rgb=0.008516
